In [2]:
!pip install gradio textblob

In [3]:
import gradio as gr
import re
from textblob import TextBlob

In [4]:
def analyze_interview(question, answer):

    if not answer.strip():
        return "⚠️ Please enter your answer.", "", "", "", ""

    # -------------------------
    # Text Preprocessing
    # -------------------------
    clean_text = answer.lower()
    clean_text = re.sub(r'[^a-zA-Z\s]', '', clean_text)

    # -------------------------
    # Word Count
    # -------------------------
    words = clean_text.split()
    word_count = len(words)

    # -------------------------
    # Keywords
    # -------------------------
    keywords = [
        "computer science",
        "python",
        "java",
        "programming",
        "artificial intelligence",
        "machine learning",
        "project",
        "skills",
        "experience",
        "student"
    ]

    found_keywords = []

    for keyword in keywords:
        if keyword in clean_text:
            found_keywords.append(keyword)

    keyword_score = (
        len(found_keywords) / len(keywords)
    ) * 100

    # -------------------------
    # Sentiment Analysis
    # -------------------------
    blob = TextBlob(answer)
    polarity = blob.sentiment.polarity

    if polarity > 0:
        sentiment = "Positive 😊"
    elif polarity < 0:
        sentiment = "Negative 😟"
    else:
        sentiment = "Neutral 😐"

    sentiment_score = ((polarity + 1) / 2) * 100

    # -------------------------
    # Length Score
    # -------------------------
    length_score = min(
        (word_count / 30) * 100,
        100
    )

    # -------------------------
    # Final Score
    # -------------------------
    final_score = (
        0.5 * keyword_score +
        0.3 * length_score +
        0.2 * sentiment_score
    )

    # -------------------------
    # Feedback
    # -------------------------
    feedback = []

    if word_count < 15:
        feedback.append(
            "⚠️ Your answer is too short. Add more details."
        )
    else:
        feedback.append(
            "✅ Good answer length."
        )

    if keyword_score < 30:
        feedback.append(
            "⚠️ Add more relevant skills, projects or experience."
        )
    else:
        feedback.append(
            "✅ Good use of relevant keywords."
        )

    if polarity > 0:
        feedback.append(
            "✅ Your answer has a positive tone."
        )
    elif polarity == 0:
        feedback.append(
            "ℹ️ Try to sound more confident and enthusiastic."
        )
    else:
        feedback.append(
            "⚠️ Try to maintain a more positive tone."
        )

    if final_score >= 75:
        overall = "🌟 Excellent interview response!"
    elif final_score >= 50:
        overall = "👍 Good response, but there is room for improvement."
    else:
        overall = "⚠️ Your answer needs improvement."

    # -------------------------
    # Results
    # -------------------------
    result = f"""
### 📊 Interview Analysis

**Question:**
{question}

**Answer:**
{answer}

---

**Word Count:** {word_count}

**Keyword Score:** {keyword_score:.1f}%

**Sentiment:** {sentiment}

**Final Score:** {final_score:.1f}/100

---

### 💡 Feedback

{chr(10).join(feedback)}

### 🏆 Overall Result

{overall}
"""

    keyword_result = ", ".join(found_keywords) if found_keywords else "No relevant keywords found."

    return (
        result,
        keyword_result,
        f"{word_count} words",
        sentiment,
        f"{final_score:.1f}/100"
    )

In [5]:
app = gr.Interface(
    fn=analyze_interview,

    inputs=[
        gr.Textbox(
            label="Interview Question",
            value="Tell me about yourself"
        ),

        gr.Textbox(
            label="Your Interview Answer",
            placeholder="Type your answer here...",
            lines=8
        )
    ],

    outputs=[
        gr.Markdown(label="Analysis"),
        gr.Textbox(label="Relevant Keywords"),
        gr.Textbox(label="Word Count"),
        gr.Textbox(label="Sentiment"),
        gr.Textbox(label="Final Score")
    ],

    title="🎯 AI Interview Analyzer",

    description=(
        "Enter an interview question and your answer. "
        "The system uses NLP to analyze your response."
    ),

    examples=[
        [
            "Tell me about yourself",
            "My name is Kavya. I am a computer science student. "
            "I know Python and Java. I have worked on several "
            "projects and I am interested in artificial intelligence "
            "and machine learning."
        ],
        [
            "What are your strengths?",
            "My strengths are programming, problem solving and "
            "learning new technologies. I enjoy working on projects "
            "and improving my technical skills."
        ]
    ]
)

In [6]:
app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4591e1a25dc562f14b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
